# 01 BEHACOM Data Audit

Cel: wykonać pierwszy, strukturalny audyt lokalnego zbioru BEHACOM bez ładowania pełnych plików CSV do pamięci.

Ten notebook jest punktem startowym. Na tym etapie nie definiujemy jeszcze BWCI i nie budujemy cech modelowych. Najpierw sprawdzamy, czy dane są kompletne, jak duże są pliki, ile mają wierszy i kolumn oraz które grupy kolumn dominują schemat.

## Założenia metodologiczne

- Analizujemy behawioralną ciągłość pracy, nie stan psychiczny użytkownika.
- BEHACOM nie zawiera etykiet `focused / distracted`; przyszła zmienna docelowa będzie proxy.
- Wczesny audyt powinien unikać pełnego ładowania 12k kolumn na plik.
- Digrafy traktujemy jako osobną, ciężką rodzinę cech, którą włączymy dopiero świadomie.

In [5]:
from __future__ import annotations

import csv
import re
from collections import Counter
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/BEHACOM/Behacom")
ENCODING = "latin-1"

COLUMN_PREFIX_GROUPS: dict[str, str] = {
  "word_length": "word_length_",
  "keystrokes_key": "keystrokes_key_",
  "press_release_average_key": "press_release_average_",
  "digraph_counter": "digraph_counter_",
  "digraph_average_time": "digraph_average_time_",
  "click_speed_average": "click_speed_average_",
  "click_speed_stddev": "click_speed_stddev_",
  "mouse_action_counter": "mouse_action_counter_",
  "mouse_position_histogram": "mouse_position_histogram_",
  "mouse_movement_direction_histogram": "mouse_movement_direction_histogram_",
  "mouse_movement_length_histogram": "mouse_movement_length_histogram_",
  "mouse_average_movement_speed_direction": "mouse_average_movement_speed_direction_",
}

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)

In [6]:
def user_sort_key(path: Path) -> int:
  """ Get a numeric sort key from a BEHACOM user path.

    Args:
      path: Path to a user CSV file.

    Returns:
      int: Numeric user id when present, otherwise a large fallback value.
  """
  match = re.search(r"User(\d+)", path.parent.name)
  return int(match.group(1)) if match else 10_000


def discover_csv_files(data_dir: Path) -> list[Path]:
  """ Discover BEHACOM user CSV files.

    Args:
      data_dir: Path to the BEHACOM directory containing user subdirectories.

    Returns:
      list[Path]: CSV file paths sorted by numeric user id.
  """
  return sorted(data_dir.glob("User*/User*_BEHACOM.csv"), key=user_sort_key)


def read_header_and_first_row(path: Path) -> tuple[list[str], list[str]]:
  """ Read a CSV header and its first data row.

    Args:
      path: Path to a BEHACOM CSV file.

    Returns:
      tuple[list[str], list[str]]: Header columns and first row fields.
  """
  with path.open("r", encoding=ENCODING, newline="") as handle:
    reader = csv.reader(handle)
    header = next(reader)
    first_row = next(reader, [])
  return header, first_row


def count_rows(path: Path) -> int:
  """ Count data rows in a CSV file without loading it into memory.

    Args:
      path: Path to a BEHACOM CSV file.

    Returns:
      int: Number of data rows excluding the header.
  """
  line_count = 0
  with path.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
      line_count += chunk.count(b"\n")
  return max(line_count - 1, 0)


def extract_user_value(header: list[str], row: list[str]) -> str | None:
  """ Extract the USER value from a BEHACOM row.

    Args:
      header: CSV header columns.
      row: CSV row fields.

    Returns:
      str | None: User value when available, otherwise None.
  """
  try:
    user_index = header.index("USER")
  except ValueError:
    return None
  return row[user_index] if user_index < len(row) else None


def classify_columns(header: list[str]) -> Counter[str]:
  """ Classify BEHACOM columns into coarse feature groups.

    Args:
      header: CSV header columns.

    Returns:
      Counter[str]: Column counts by feature group.
  """
  groups: Counter[str] = Counter()
  for column in header:
    if column == "press_release_average_interval":
      groups["core_or_other"] += 1
      continue
    for group, prefix in COLUMN_PREFIX_GROUPS.items():
      if column.startswith(prefix):
        groups[group] += 1
        break
    else:
      groups["core_or_other"] += 1
  return groups

## Audyt plików

Poniższa komórka czyta tylko nagłówki, pierwszy wiersz i liczy wiersze strumieniowo. Nie ładuje całych CSV do `DataFrame`.

In [7]:
files = discover_csv_files(DATA_DIR)
if not files:
  raise FileNotFoundError(f"No BEHACOM CSV files found under {DATA_DIR.resolve()}")

reference_header, _ = read_header_and_first_row(files[0])

file_rows: list[dict[str, object]] = []
for path in files:
  header, first_row = read_header_and_first_row(path)
  file_rows.append(
    {
      "user_dir": path.parent.name,
      "file": path.name,
      "size_mb": round(path.stat().st_size / 1_000_000, 2),
      "rows": count_rows(path),
      "columns": len(header),
      "first_user": extract_user_value(header, first_row),
      "first_row_fields": len(first_row),
      "header_ok": header == reference_header,
    }
  )

files_df = pd.DataFrame(file_rows)
files_df

,user_dir,file,size_mb,rows,columns,first_user,first_row_fields,header_ok
0,User0,User0_BEHACOM.csv,219.91,6059,12051,0,12051,True
1,User1,User1_BEHACOM.csv,1530.44,42281,12051,1,12051,True
2,User2,User2_BEHACOM.csv,6.85,179,12051,2,12051,True
3,User3,User3_BEHACOM.csv,116.98,3221,12051,3,12051,True
4,User4,User4_BEHACOM.csv,367.59,10114,12051,4,12051,True
5,User5,User5_BEHACOM.csv,77.41,2128,12051,5,12051,True
6,User6,User6_BEHACOM.csv,48.55,1332,12051,6,12051,True
7,User7,User7_BEHACOM.csv,2018.92,55778,12051,7,12051,True
8,User8,User8_BEHACOM.csv,196.35,5404,12051,8,12051,True
9,User9,User9_BEHACOM.csv,287.60,7920,12051,9,12051,True


In [8]:
summary = {
  "csv_files": len(files_df),
  "total_size_mb": round(files_df["size_mb"].sum(), 2),
  "total_rows": int(files_df["rows"].sum()),
  "rows_without_user2": int(files_df.loc[files_df["first_user"] != "2", "rows"].sum()),
  "columns_per_file": int(files_df["columns"].iloc[0]),
  "headers_consistent": bool(files_df["header_ok"].all()),
}

pd.Series(summary)

csv_files                  12
total_size_mb         6054.68
total_rows             167058
rows_without_user2     166879
columns_per_file        12051
headers_consistent       True
dtype: object

## Audyt grup kolumn

In [9]:
column_groups = classify_columns(reference_header)
column_groups_df = (
  pd.DataFrame(sorted(column_groups.items()), columns=["group", "columns"])
  .sort_values("columns", ascending=False)
  .reset_index(drop=True)
)

column_groups_df

,group,columns
0,digraph_counter,5878
1,digraph_average_time,5878
2,keystrokes_key,105
3,press_release_average_key,105
4,core_or_other,31
5,word_length,11
6,mouse_position_histogram,9
7,mouse_average_movement_speed_direction,8
8,mouse_movement_direction_histogram,8
9,mouse_action_counter,7


## Wnioski z audytu

- Zbiór jest lokalnie dostępny i ma spójny schemat między użytkownikami.
- Pełne ładowanie CSV nie jest dobrym domyślnym trybem pracy, ponieważ każdy plik ma ponad 12 tys. kolumn.
- `User2` ma bardzo mało danych i powinien być wykluczony z większości dalszych analiz.
- Digrafy dominują liczbę kolumn, więc pierwszy właściwy EDA powinien używać selektywnego wczytywania kolumn bez digrafów.

Następny krok: selektywne wczytanie podstawowych kolumn, policzenie okien aktywnych/idle per użytkownik oraz pierwsza kontrola jakości wartości odstających.